In [1]:
import yaml
import torch
import numpy as np
from dotmap import DotMap
import matplotlib.pyplot as plt

from pud.collector import Collector
from pud.visualize import plot_agent_paths
from pud.vision_agent import LagVisionUVFDDPG
from pud.utils import set_global_seed, set_env_seed
from pud.algos.constrained_collector import ConstrainedCollector
from pud.policies import VisualSearchPolicy, VisualMultiAgentSearchPolicy
from pud.envs.safe_pointenv.pb_sampler import load_pb_set, sample_pbs_by_agent
from pud.envs.habitat_navigation_env import GoalConditionedHabitatPointWrapper
from pud.envs.safe_habitatenv.safe_habitat_wrappers import safe_habitat_env_load_fn
from pud.envs.safe_habitatenv.safe_habitat_wrappers import SafeGoalConditionedHabitatPointWrapper, SafeGoalConditionedHabitatPointQueueWrapper

#### Load the evaluation configuration

In [2]:
illustration_pb_file = "pud/envs/safe_habitatenv/illustration_set/SC2_Staging_08.txt"
unconstrained_ckpt_file = "/home/mers-pluto/job_26896380_visual_cost_correct_flag/2024-08-28-03-58-20/ckpt/ckpt_0997500"
config_file = "/home/mers-pluto/job_26896380_visual_cost_correct_flag/2024-08-28-03-58-20/lag/2024-08-30-06-17-29/bk/config.yaml"
constrained_ckpt_file = "/home/mers-pluto/job_26896380_visual_cost_correct_flag/2024-08-28-03-58-20/lag/2024-08-30-06-17-29/ckpt/ckpt_0192500"

with open(config_file, 'r') as f:
    config = yaml.safe_load(f)
config = DotMap(config)

# User defined parameters for evaluation
config.device = "cuda:0"
config.agent_cost_kwargs.cost_limit = 1.5
config.replay_buffer.max_size = 1000

set_global_seed(config.seed)

#### Load the evaluation environment

In [3]:
gym_env_wrappers = []
gym_env_wrapper_kwargs = []
for wrapper_name in config.wrappers:
    if wrapper_name == "GoalConditionedHabitatPointWrapper":
        gym_env_wrappers.append(GoalConditionedHabitatPointWrapper)
        gym_env_wrapper_kwargs.append(config.wrappers[wrapper_name].toDict())
    elif wrapper_name == "SafeGoalConditionedHabitatPointWrapper":
        gym_env_wrappers.append(SafeGoalConditionedHabitatPointWrapper)
        gym_env_wrapper_kwargs.append(config.wrappers[wrapper_name].toDict())
    elif wrapper_name == "SafeGoalConditionedHabitatPointQueueWrapper":
        gym_env_wrappers.append(SafeGoalConditionedHabitatPointQueueWrapper)
        gym_env_wrapper_kwargs.append(config.wrappers[wrapper_name].toDict())

eval_env = safe_habitat_env_load_fn(
    env_kwargs=config.env.toDict(),
    cost_f_args=config.cost_function.toDict(),
    cost_limit=config.agent_cost_kwargs.cost_limit,
    max_episode_steps=config.time_limit.max_episode_steps,
    gym_env_wrappers=gym_env_wrappers,  # type: ignore
    wrapper_kwargs=gym_env_wrapper_kwargs,
    terminate_on_timeout=True,
    )
set_env_seed(eval_env, config.seed + 2)

Renderer: NVIDIA GeForce RTX 4090/PCIe/SSE2 by NVIDIA Corporation
OpenGL version: 4.6.0 NVIDIA 550.107.02
Using optional features:
    GL_ARB_vertex_array_object
    GL_ARB_separate_shader_objects
    GL_ARB_robustness
    GL_ARB_texture_storage
    GL_ARB_texture_view
    GL_ARB_framebuffer_no_attachments
    GL_ARB_invalidate_subdata
    GL_ARB_texture_storage_multisample
    GL_ARB_multi_bind
    GL_ARB_direct_state_access
    GL_ARB_get_texture_sub_image
    GL_ARB_texture_filter_anisotropic
    GL_KHR_debug
    GL_KHR_parallel_shader_compile
    GL_NV_depth_buffer_float
Using driver workarounds:
    no-forward-compatible-core-context
    nv-egl-incorrect-gl11-function-pointers
    no-layout-qualifiers-on-old-glsl
    nv-zero-context-profile-mask
    nv-implementation-color-read-format-dsa-broken
    nv-cubemap-inconsistent-compressed-image-size
    nv-cubemap-broken-full-compressed-image-query
    nv-compressed-block-size-in-bits
[INFO] Calling the APSP construction function


[17:51:21:743515]:[Error]:[Metadata] SceneDatasetAttributesManager.cpp(304)::validateMap : `navmesh_instances` Value : `navmeshes/Baked_sc4_staging_00.navmesh` not found on disk as absolute path or relative to `external_data/replica_cad/replica_cad_baked_lighting`
[17:51:21:743531]:[Error]:[Metadata] SceneDatasetAttributesManager.cpp(304)::validateMap : `navmesh_instances` Value : `navmeshes/Baked_sc4_staging_01.navmesh` not found on disk as absolute path or relative to `external_data/replica_cad/replica_cad_baked_lighting`
[17:51:21:743535]:[Error]:[Metadata] SceneDatasetAttributesManager.cpp(304)::validateMap : `navmesh_instances` Value : `navmeshes/Baked_sc4_staging_02.navmesh` not found on disk as absolute path or relative to `external_data/replica_cad/replica_cad_baked_lighting`
[17:51:21:743539]:[Error]:[Metadata] SceneDatasetAttributesManager.cpp(304)::validateMap : `navmesh_instances` Value : `navmeshes/Baked_sc4_staging_03.navmesh` not found on disk as absolute path or relativ

  0%|          | 0/189 [00:00<?, ?it/s]

APSP construction time in (s):  0.020038366317749023
[INFO] Skipping the reset in HabitatNavigationEnv.__init__ because setup is not ready yet
[INFO] SafeHabitatNavigationEnv Setup: 0.00679469108581543 s


#### Load the inference agent

In [4]:
config.agent["action_dim"] = eval_env.action_space.shape[0]  # type: ignore
config.agent["max_action"] = float(eval_env.action_space.high[0])  # type: ignore

agent = LagVisionUVFDDPG(
    width=config.env.simulator_settings.width,
    height=config.env.simulator_settings.height,
    in_channels=4,
    act_fn=torch.nn.SELU,
    encoder="VisualEncoder",
    device=config.device,
    **config.agent.toDict(),
    cost_kwargs=config.agent_cost_kwargs.toDict(),
)

if len(constrained_ckpt_file) > 0:
    agent.load_state_dict(torch.load(constrained_ckpt_file))
else:
    agent.load_state_dict(torch.load(unconstrained_ckpt_file))
agent.to(torch.device(config.device))
agent.eval()

LagVisionUVFDDPG(
  (actor): VisualGoalConditionedActor(
    (encoder): VisualEncoder(
      (conv_net): Sequential(
        (0): Conv2d(4, 16, kernel_size=(8, 8), stride=(4, 4))
        (1): SELU()
        (2): Conv2d(16, 32, kernel_size=(4, 4), stride=(4, 4))
        (3): SELU()
        (4): Flatten(start_dim=1, end_dim=-1)
      )
      (l1): Linear(in_features=128, out_features=256, bias=True)
    )
    (l1): Linear(in_features=512, out_features=256, bias=True)
    (l2): Linear(in_features=256, out_features=256, bias=True)
    (l3): Linear(in_features=256, out_features=2, bias=True)
  )
  (actor_target): VisualGoalConditionedActor(
    (encoder): VisualEncoder(
      (conv_net): Sequential(
        (0): Conv2d(4, 16, kernel_size=(8, 8), stride=(4, 4))
        (1): SELU()
        (2): Conv2d(16, 32, kernel_size=(4, 4), stride=(4, 4))
        (3): SELU()
        (4): Flatten(start_dim=1, end_dim=-1)
      )
      (l1): Linear(in_features=128, out_features=256, bias=True)
    )
    (l

#### Sample a replay buffer to form our search graph

In [5]:
rb_vec_grid, rb_vec_visual = ConstrainedCollector.sample_initial_unconstrained_states(eval_env, config.replay_buffer.max_size, habitat=True)
pdist = agent.get_pairwise_dist(rb_vec_visual, aggregate=None)  # type: ignore

#### Sample (start, goal) problems for the agent

In [6]:
if len(illustration_pb_file) > 0:
    problems = load_pb_set(file_path=illustration_pb_file, env=eval_env, agent=agent)  # type: ignore
else:
    problems = sample_pbs_by_agent(
        K=10,
        min_dist=0,
        agent=agent,  # type: ignore
        env=eval_env,  # type: ignore
        target_val=10,
        num_states=100,
        ensemble_agg="mean",
        use_uncertainty=False,
        max_dist=eval_env.max_goal_dist,  # type: ignore
    )
    assert len(problems) > 0

# problems = problems[1:]
eval_env.set_use_q(True)  # type: ignore
eval_env.set_prob_constraint(1.0)  # type: ignore
eval_env.set_pbs(pb_list=problems.copy())  # type: ignore

## Single-Agent Comparisons

### Unconstrained Low-Level Policy

In [7]:
agent.load_state_dict(torch.load(unconstrained_ckpt_file))
agent.to(torch.device(config.device))
agent.eval()

eval_env.duration = 300  # type: ignore
start, goal, unconstrained_observations, _, _, _ = Collector.get_trajectory(agent, eval_env, habitat=True)

AssertionError: 

### Unconstrained Low-Level Policy with Graph Search

In [ ]:
eval_env.duration = 300  # type: ignore

search_policy = VisualSearchPolicy(agent, (rb_vec_grid, rb_vec_visual), pdist=pdist, open_loop=True, max_search_steps=3, no_waypoint_hopping=True,)
start, goal, unconstrained_search_observations, unconstrained_search_waypoints, _, _ = Collector.get_trajectory(search_policy, eval_env, habitat=True, input_start=start, input_goal=goal)

### Constrained Low-Level Policy

In [ ]:
agent.load_state_dict(torch.load(constrained_ckpt_file))
agent.to(torch.device(config.device))
agent.eval()

eval_env.duration = 300  # type: ignore
eval_env.set_pbs(pb_list=problems.copy())  # type: ignore

start, goal, constrained_observations, _, _ = ConstrainedCollector.get_trajectory(agent, eval_env)

### Constrained Low-Level Policy with Graph Search

In [ ]:
eval_env.duration = 300  # type: ignore
eval_env.set_pbs(pb_list=problems.copy())  # type: ignore

constrained_search_policy = ConstrainedSearchPolicy(agent, rb_vec, pdist=pdist, pcost=pcost, open_loop=True, no_waypoint_hopping=True, max_cost_limit=config.agent.cost_limit)
start, goal, constrained_search_observations, constrained_search_waypoints, _ = ConstrainedCollector.get_trajectory(constrained_search_policy, eval_env)

#### Plot the single-agent comparison

In [8]:
height, width = eval_env.walls.shape
normalization_factor = np.array([height, width])

start = start[0] / normalization_factor
goal = goal[0] / normalization_factor

unconstrained_observations = np.array([obs[0] / normalization_factor for obs in unconstrained_observations])
unconstrained_search_observations = np.array([obs[0] / normalization_factor for obs in unconstrained_search_observations])
unconstrained_search_waypoints = np.array([wp[0] / normalization_factor for wp in unconstrained_search_waypoints])

In [ ]:
from pud.envs.habitat_navigation_env import plot_wall


fig, axs = plt.subplots(1, 4, figsize=(12, 4))

for ax in axs:
    ax = plot_wall(eval_env.walls, ax)
    # ax.imshow(1.0 - eval_env.walls, cmap="binary", interpolation="nearest", alpha=0.5, extent=[0, 1, 0, 1])
    ax.set_xticks([])
    ax.set_yticks([])

axs[0] = plot_agent_paths(0, start, goal, unconstrained_observations, "Unconstrained\nPolicy", axs[0], use_agent_id=False)
axs[1] = plot_agent_paths(0, start, goal, unconstrained_search_observations, "Unconstrained\nSearch Policy", axs[1], wps=unconstrained_search_waypoints, use_agent_id=False)

plt.legend(loc="lower center", bbox_to_anchor=(-1.3, -0.25), ncol=4, fontsize=12)
_ = plt.suptitle("Single-Agent Trajectory Comparison", fontsize=20)

## Multi-Agent Comparisons

### Unconstrained Low-Level Policy

In [ ]:
n_agents = 4
agent.load_state_dict(torch.load(unconstrained_ckpt_file))
agent.to(torch.device(config.device))
agent.eval()

eval_env.duration = 300  # type: ignore
start, goal, unconstrained_observations, _, _, _ = Collector.get_trajectories(agent, eval_env, n_agents, habitat=True)

### Unconstrained Low-Level Policy with Graph Search

In [ ]:
eval_env.duration = 300  # type: ignore

ma_search_policy = VisualMultiAgentSearchPolicy(agent, (rb_vec_grid, rb_vec_visual), n_agents, pdist=pdist, open_loop=True, max_search_steps=3, no_waypoint_hopping=True)
start, goal, unconstrained_search_observations, unconstrained_search_waypoints, _, _ = Collector.get_trajectories(ma_search_policy, eval_env, n_agents, habitat=True, input_starts=start, input_goals=goal)

### Constrained Low-Level Policy

In [ ]:
agent.load_state_dict(torch.load(constrained_ckpt_file))
agent.to(torch.device(config.device))
agent.eval()

eval_env.duration = 300  # type: ignore
eval_env.set_pbs(pb_list=problems.copy())  # type: ignore

start, goal, constrained_observations, _, _ = ConstrainedCollector.get_trajectories(agent, eval_env, n_agents)

### Constrained Low-Level Policy with Graph Search

In [ ]:
eval_env.duration = 300  # type: ignore
eval_env.set_pbs(pb_list=problems.copy())  # type: ignore

constrained_ma_search_policy = ConstrainedMultiAgentSearchPolicy(agent, rb_vec, n_agents, pdist=pdist, pcost=pcost, open_loop=True, no_waypoint_hopping=True, max_cost_limit=config.agent.cost_limit)
start, goal, constrained_search_observations, constrained_search_waypoints, _ = ConstrainedCollector.get_trajectories(constrained_ma_search_policy, eval_env, n_agents)

#### Plot the multi-agent comparison

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(12, 4))

for ax in axs:
    ax = plot_wall(eval_env.walls, ax)
    ax.set_xticks([])
    ax.set_yticks([])

for agent_id in range(n_agents):

    agent_start = start[agent_id][0] / normalization_factor
    agent_goal = goal[agent_id][0] / normalization_factor

    agent_unconstrained_observations = np.array([obs[0] / normalization_factor for obs in unconstrained_observations[agent_id]])
    agent_unconstrained_search_observations = np.array([obs[0] / normalization_factor for obs in unconstrained_search_observations[agent_id]])
    agent_unconstrained_search_waypoints = np.array([wp[0] / normalization_factor for wp in unconstrained_search_waypoints[agent_id]])

    axs[0] = plot_agent_paths(agent_id, agent_start, agent_goal, agent_unconstrained_observations, "Unconstrained\nPolicy", axs[0])
    axs[1] = plot_agent_paths(agent_id, agent_start, agent_goal, agent_unconstrained_search_observations, "Unconstrained\nSearch Policy", axs[1], wps=agent_unconstrained_search_waypoints)

plt.legend(loc="lower center", bbox_to_anchor=(-1.3, -0.7), ncol=4, fontsize=12)
_ = plt.suptitle("Multi-Agent Trajectory Comparison", fontsize=24)